# Email security of Zurich SMEs

Data: `results/mail_extensive.csv`, produced by the passive DNS scan

```
.venv/bin/python -m sme_scan.scan zh_sme_domains_extensive.csv --out results/mail_extensive.csv
```

joined with the company list (`Category`, `Municipality`) from `zh_sme_domains_extensive.csv`.

## 1. Load and join

The scan output only knows domains; the category lives in the source list. We join on `domain`,
cleaned with **the same `clean_domain` function the scanner used**. If the notebook had its own
cleanup logic, the two could quietly drift apart and rows would stop matching.

`indicator=True` adds a `_merge` column saying which side each row came from: a cheap way to
check that nothing got lost in the join.

In [8]:
import pandas as pd
from sme_scan.scan import clean_domain

src = pd.read_csv("zh_sme_domains_extensive.csv")
res = pd.read_csv("results/mail_extensive.csv")

src["domain"] = src["Website"].map(clean_domain)

df = src.merge(res, on="domain", how="left", indicator=True)
df["_merge"].value_counts()

_merge
both          193
left_only       0
right_only      0
Name: count, dtype: int64

Keep only companies we actually have a result for. `unknown` means the DNS lookup failed,
which says nothing about the company, so it must not be counted as "unprotected".

In [10]:
df = df[(df["_merge"] == "both") & (df["verdict"] != "unknown")].drop(columns="_merge")
print(len(df), "companies with a usable scan result")

193 companies with a usable scan result


## 2. Yes/no columns for each measure

The trick that makes the rest easy: **the mean of a True/False column is the share of True**,
because Python counts `True` as 1 and `False` as 0. So `df["has_dmarc"].mean()` *is* the DMARC adoption rate.

Some columns first need building:
- **Empty CSV cells** load as `NaN`, so "has an SPF record" is `.notna()`.
- **Columns with blanks** (e.g. `spf_too_many_lookups` is blank when there's no SPF at all) load as
  `object` dtype, holding `True`, `False` and `NaN`. `.eq(True)` turns that into a clean boolean
  column where blank counts as False. (Don't use `.astype(bool)` here: `NaN` would become `True`.)

In [11]:
df["has_spf"] = df["spf_record"].notna()
df["has_dmarc"] = df["verdict"].ne("no DMARC")
df["dmarc_enforced"] = df["verdict"].eq("protected")
df["dkim_found"] = df["dkim_selectors"].notna()
for col in ["dmarc_rua", "mta_sts", "spf_too_many_lookups"]:
    df[col] = df[col].eq(True)

metrics = ["has_spf", "has_dmarc", "dmarc_enforced", "dmarc_rua", "dkim_found"]
(df[metrics].mean() * 100).round(1)

has_spf           85.0
has_dmarc         38.3
dmarc_enforced    20.2
dmarc_rua          8.3
dkim_found        37.8
dtype: float64

## 3. Who runs the mail, per category?

`pd.crosstab` counts rows for every combination of two columns. `margins=True` adds totals.

In [12]:
pd.crosstab(df["Category"], df["provider_type"], margins=True, margins_name="total")

provider_type,cloud suite,gateway,hoster,none,self-hosted,unknown,total
Category,,,,,,,
Arztpraxis / Zahnarztpraxis,7,2,23,0,1,4,37
Detailhandel,9,1,26,1,0,2,39
Restaurant / Gastro,8,2,23,4,0,2,39
Schreinerei / Zimmerei / Bauunternehmen,13,2,18,2,0,3,38
Treuhand / Fiduciary,24,4,9,0,0,3,40
total,61,11,99,7,1,14,193


Same table as **percent of each row** (`normalize="index"`): read across a row to see how one
category splits between provider types. Rows add up to 100.

In [13]:
(pd.crosstab(df["Category"], df["provider_type"], normalize="index") * 100).round(0)

provider_type,cloud suite,gateway,hoster,none,self-hosted,unknown
Category,,,,,,
Arztpraxis / Zahnarztpraxis,19.0,5.0,62.0,0.0,3.0,11.0
Detailhandel,23.0,3.0,67.0,3.0,0.0,5.0
Restaurant / Gastro,21.0,5.0,59.0,10.0,0.0,5.0
Schreinerei / Zimmerei / Bauunternehmen,34.0,5.0,47.0,5.0,0.0,8.0
Treuhand / Fiduciary,60.0,10.0,22.0,0.0,0.0,8.0


## 4. Protection rates by category and by provider type

`groupby(...).mean()` on the boolean columns gives the percentage per group. The `n` column
matters: with ~38 companies per category, **one company is about 2.6 percentage points**, so
differences of a few points between categories are noise, not findings.

In [14]:
def rates(by):
    """Percent of companies meeting each measure, per group, plus group size n."""
    grouped = df.groupby(by)
    table = (grouped[metrics].mean() * 100).round(0)
    table["n"] = grouped.size()
    return table.sort_values("dmarc_enforced", ascending=False)

rates("Category")

,has_spf,has_dmarc,dmarc_enforced,dmarc_rua,dkim_found,n
Category,,,,,,
Detailhandel,85.0,51.0,31.0,10.0,49.0,39
Treuhand / Fiduciary,95.0,35.0,22.0,5.0,40.0,40
Schreinerei / Zimmerei / Bauunternehmen,87.0,34.0,18.0,11.0,34.0,38
Arztpraxis / Zahnarztpraxis,70.0,32.0,16.0,3.0,22.0,37
Restaurant / Gastro,87.0,38.0,13.0,13.0,44.0,39


In [ ]:
rates("provider_type")

## 5. Category × provider type

Is the provider effect the same in every category? We keep only the two big provider types,
since the others are too small to split further, and show the **enforcement rate** next to the
**count** in each cell. Always look at both together: 100% of 2 companies is an anecdote.

In [ ]:
main = df[df["provider_type"].isin(["cloud suite", "hoster"])]
pd.pivot_table(
    main,
    index="Category",
    columns="provider_type",
    values="dmarc_enforced",
    aggfunc=["mean", "count"],
).round(2)